# Q4 — AMZN Earnings Surprise: Median 2-Day Return & Correlation
**Homework:** `homework1.md:97-125` — *Median 2-day % change after positive AMZN earnings surprises + correlation.*

**Novice:** `get_earnings_dates(limit=25)` → 25 rows (1 future NaN), `Close_Day3/Close_Day1-1` centered on Day2.

**Answer:** **Median 0.35%** (0.003528) for 20 positives, **corr 0.331** (positive) / 0.219 (all).


## 4.1 Earnings dates (yfinance)

- `yf.Ticker('AMZN').get_earnings_dates(limit=25)` → 2020-10-29 onward, 1 future NaN
- Check shape, columns.

In [ ]:
import yfinance as yf, pandas as pd
ticker="AMZN"
earn = yf.Ticker(ticker).get_earnings_dates(limit=25)
print(earn.shape, earn.columns.tolist())
print(earn.head().to_string())
print(f"Future NaN: {earn['Reported EPS'].isna().sum()}")

## 4.2 Price history and 2-day returns

- Download 2019-present for windows
- For 3 consecutive days `Day1,Day2,Day3`: `ret = Close_Day3/Close_Day1 -1` stored at `Day2`
- Novice: `shift(1)` and `shift(-1)` handle consecutive trading days.

In [ ]:
close = yf.download(ticker, start="2019-01-01", progress=False, auto_adjust=False)
if isinstance(close.columns, pd.MultiIndex): close = close["Close"][ticker]
else: close = close["Close"]
close = close.dropna()
tmp = pd.DataFrame({"Close":close})
tmp["2d_ret"] = tmp["Close"].shift(-1) / tmp["Close"].shift(1) -1
tmp["Day2_date"] = pd.to_datetime(tmp.index.normalize())
print(tmp.head(3).to_string())
print(f"Rows {len(tmp)}, example 2d_ret {tmp['2d_ret'].iloc[5]:.4f}")

## 4.3 Merge earnings → 2-day return

- Normalize earnings timestamp to date, match to `Day2_date` exact; if weekend, next trading day
- Build DataFrame with `surprise` and `ret_2d`.

In [ ]:
ret_by_day = dict(zip(tmp["Day2_date"], tmp["2d_ret"]))
records=[]
for idx,row in earn.iterrows():
    ed = pd.Timestamp(idx.date()).normalize()
    ret = ret_by_day.get(ed, float("nan"))
    matched=ed if pd.notna(ret) else None
    if pd.isna(ret):
        nxt = min([d for d in ret_by_day if d>=ed], default=None)
        if nxt is not None: ret,matched = ret_by_day[nxt],nxt
    records.append({"earn_date":ed.date(), "surprise":row["Surprise(%)"], "ret_2d":ret})
res = pd.DataFrame(records)
print(res.to_string(index=False))

## 4.4 Median (positive) and correlation via `pd.corr()`

- Filter `surprise>0` → 20 rows
- Hint: `positive[["surprise","ret_2d"]].corr()`

In [ ]:
positive = res[(res["surprise"]>0) & res["surprise"].notna() & res["ret_2d"].notna()]
print(f"Positive: {len(positive)}/25")
print(f"Median {positive['ret_2d'].median()*100:.4f}%")
print(positive[["surprise","ret_2d"]].corr().to_string())
print(f"Answer Q4 median {positive['ret_2d'].median()*100:.2f}% corr {positive[['surprise','ret_2d']].corr().iloc[0,1]:.3f}")